In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parents[1]
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed path exists:", DATA_PROCESSED.exists())

Project root: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction
Processed path exists: True


In [2]:
ipo = pd.read_csv(DATA_PROCESSED / "ipo_master_stage1.csv")

ipo["company_name"] = (
    ipo["COMPANY NAME"]
    .str.lower()
    .str.strip()
)

ipo["listing_date"] = pd.to_datetime(
    ipo["DATE OF LISTING"],
    errors="coerce",
    utc=True
)

ipo = ipo[[
    "company_name",
    "listing_date"
]].dropna(subset=["listing_date"])

ipo.shape

(1179, 2)

In [3]:
market = pd.read_csv(
    DATA_PROCESSED / "market_features.csv",
    parse_dates=["listing_date"]
)

market["company_name"] = market["company_name"].str.lower().str.strip()

market.shape

(1156, 11)

In [4]:
sector = pd.read_csv(
    DATA_PROCESSED / "sector_and_regime_features.csv",
    parse_dates=["listing_date"]
)

sector["company_name"] = sector["company_name"].str.lower().str.strip()

sector.shape

(3577, 10)

In [5]:
market_clean = (
    market
    .sort_values("listing_date")
    .drop_duplicates(
        subset=["company_name", "listing_date"],
        keep="last"
    )
)

market_clean.shape

(971, 11)

In [6]:
sector_clean = (
    sector
    .sort_values("listing_date")
    .drop_duplicates(
        subset=["company_name", "listing_date"],
        keep="last"
    )
)

sector_clean.shape

(987, 10)

In [7]:
ipo_clean = (
    ipo
    .sort_values("listing_date")
    .drop_duplicates(
        subset=["company_name", "listing_date"],
        keep="last"
    )
)

ipo_clean.shape

(987, 2)

In [8]:
df = ipo_clean.merge(
    market_clean,
    on=["company_name", "listing_date"],
    how="left",
    validate="many_to_one"
)

In [9]:
df = df.merge(
    sector_clean,
    on=["company_name", "listing_date"],
    how="left",
    validate="many_to_one"
)

In [10]:
df.isna().mean().sort_values(ascending=False).head(10)

high_volatility_x         0.016211
market_return_1d          0.016211
market_return_7d          0.016211
market_return_30d         0.016211
market_vol_30d            0.016211
recent_market_trend       0.016211
bull_market               0.016211
bear_market               0.016211
days_from_market_start    0.016211
market_regime             0.000000
dtype: float64

In [11]:
df.duplicated(
    subset=["company_name", "listing_date"]
).any()

np.False_

In [13]:
df = ipo_clean.merge(
    market_clean,
    on=["company_name", "listing_date"],
    how="left",
    validate="many_to_one"
)

df = df.merge(
    sector_clean,
    on=["company_name", "listing_date"],
    how="left",
    validate="many_to_one"
)

df.shape

(987, 19)

In [17]:
# Load section-level embeddings
sec_emb = pd.read_parquet(
    DATA_PROCESSED / "drhp_embeddings" / "drhp_section_embeddings.parquet"
)

sec_emb["company_name"] = (
    sec_emb["company_name"]
    .str.lower()
    .str.strip()
)

sec_emb.shape

(842, 5)

In [18]:
def mean_stack(arrs):
    return np.mean(np.vstack(arrs), axis=0)

ipo_emb = (
    sec_emb
    .groupby("company_name", as_index=False)
    .agg(
        finbert_all=("finbert_embedding", mean_stack),
        sbert_all=("sbert_embedding", mean_stack),
        sections_present=("section", "nunique")
    )
)

ipo_emb.shape

(272, 4)

In [19]:
ipo_emb["company_name"].is_unique

True

In [20]:
df_final = df.merge(
    ipo_emb,
    on="company_name",
    how="left",
    validate="one_to_one"
)

df_final.shape

(987, 22)

In [21]:
df_final.duplicated(
    ["company_name", "listing_date"]
).any()

np.False_

In [22]:
df_final.isna().mean().sort_values(ascending=False).head(10)

sections_present       0.883485
sbert_all              0.883485
finbert_all            0.883485
market_return_1d       0.016211
market_return_7d       0.016211
market_return_30d      0.016211
market_vol_30d         0.016211
recent_market_trend    0.016211
bull_market            0.016211
bear_market            0.016211
dtype: float64

In [24]:
# Keep market volatility, drop sector duplicate
df_final = df_final.drop(columns=["high_volatility_y"])
df_final = df_final.rename(columns={"high_volatility_x": "high_volatility"})

In [25]:
df_final.isna().mean().sort_values(ascending=False).head(10)

sections_present          0.883485
sbert_all                 0.883485
finbert_all               0.883485
bull_market               0.016211
high_volatility           0.016211
bear_market               0.016211
days_from_market_start    0.016211
recent_market_trend       0.016211
market_vol_30d            0.016211
market_return_30d         0.016211
dtype: float64

In [26]:
OUT_PATH = DATA_PROCESSED / "phase10_final_dataset.csv"
df_final.to_csv(OUT_PATH, index=False)

print("✅ Final multimodal dataset saved at:", OUT_PATH)

✅ Final multimodal dataset saved at: /Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/data/processed/phase10_final_dataset.csv
